# 8. Кластеризация тем ноутбуков по тегам
Теги, которые часто стоят вместе, относятся к одной теме.
Матрица «тег × ноутбук» → TruncatedSVD → KMeans по тегам.

In [ ]:
import sys
sys.path.append("..")

import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.cluster import KMeans
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import normalize
from src.data_loader import load

kt = load("KernelTags").merge(load("Tags"), left_on="TagId", right_on="Id")
kernels = load("Kernels").set_index("Id")

Берём самые частые теги, чтобы редкие не давали шум.

In [ ]:
TOP_TAGS = 150
top = kt["Name"].value_counts().head(TOP_TAGS).index
kt = kt[kt["Name"].isin(top)]

tag_idx = {t: i for i, t in enumerate(top)}
kernel_ids = kt["KernelId"].unique()
kernel_idx = {k: i for i, k in enumerate(kernel_ids)}
m = csr_matrix(
    (np.ones(len(kt)), (kt["Name"].map(tag_idx), kt["KernelId"].map(kernel_idx))),
    shape=(len(top), len(kernel_ids)),
)
m.shape

In [ ]:
emb = normalize(TruncatedSVD(n_components=min(20, m.shape[0] - 1), random_state=0).fit_transform(m))
N_CLUSTERS = 8
labels = KMeans(n_clusters=N_CLUSTERS, n_init=10, random_state=0).fit_predict(emb)
clusters = pd.Series(labels, index=top, name="cluster")

for c in range(N_CLUSTERS):
    print(c, ", ".join(clusters[clusters == c].index[:12]))

## Популярность ноутбуков по кластерам
Ноутбук относим к кластеру, в который попало больше всего его тегов.

In [ ]:
kt["cluster"] = kt["Name"].map(clusters)
main_cluster = kt.groupby("KernelId")["cluster"].agg(lambda s: s.mode().iloc[0])
votes = kernels["TotalVotes"].reindex(main_cluster.index)
pd.DataFrame({"cluster": main_cluster, "votes": votes}).groupby("cluster")["votes"].agg(["count", "median", "mean"])

TODO: подобрать число кластеров (силуэт), дать кластерам названия.